In [ ]:
!pip install -U "mcp[cli]"

In [ ]:
%%writefile mcp_server.py

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo Server")

@mcp.tool()
def hello() -> str:
    """Return a static greeting."""
    return "Hello from the MCP Server!"

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

In [ ]:
%%writefile mcp_client_http.py

import asyncio
from mcp import Client
from mcp.client.streamable_http import streamablehttp_client

async def main():
    async with streamablehttp_client(
        "http://127.0.0.1:8000/mcp"
    ) as (read_stream, write_stream, _):

        async with Client(read_stream, write_stream) as client:
            await client.initialize()

            result = await client.call_tool("hello", {})

            for content in result.content:
                print(content.text)

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
%%writefile mcp_server.py

from mcp.server.fastmcp import FastMCP
import random

mcp = FastMCP("Music Server")

@mcp.tool()
def hello() -> str:
    """Return a static greeting."""
    return "Hello from the MCP Server!"

@mcp.tool()
def get_song_recommendation() -> dict:
    """Return a random trending song recommendation."""
    
    songs = [
        "Blinding Lights",
        "Shape of You",
        "As It Was",
        "Flowers",
        "Levitating"
    ]

    return {
        "song": random.choice(songs),
        "status": "trending"
    }

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

In [ ]:
%%writefile mcp_song_client.py

import asyncio
from mcp import Client
from mcp.client.streamable_http import streamablehttp_client

async def main():
    async with streamablehttp_client(
        "http://127.0.0.1:8000/mcp"
    ) as (read_stream, write_stream, _):

        async with Client(read_stream, write_stream) as client:
            await client.initialize()

            result = await client.call_tool(
                "get_song_recommendation",
                {}
            )

            for content in result.content:
                print(content.text)

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
## Task 4: MCP Transport Comparison

### stdio

stdio is useful when the MCP client and server run on the same computer. The client launches the MCP server as a subprocess and communicates with it through standard input and output. It is simple, lightweight, and commonly used by desktop AI applications.

### Streamable HTTP

Streamable HTTP is useful when the MCP server runs as a separate service. The client communicates with the server through an HTTP endpoint, making it more suitable for remote or deployed applications.

### Comparison

| Feature | stdio | Streamable HTTP |
|---|---|---|
| Communication | stdin/stdout | HTTP |
| Server location | Usually local | Local or remote |
| Network required | No | Yes |
| Typical use | Desktop/local tools | Deployed services |
| Setup | Simple | More suitable for networked systems |

In [ ]:
%%writefile mcp_server_stdio.py

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Stdio Server")

@mcp.tool()
def get_song_recommendation() -> dict:
    """Return a trending song."""
    return {
        "song": "Blinding Lights",
        "status": "trending"
    }

if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
%%writefile mcp_client_stdio.py

import asyncio
import sys
from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():

    server_params = StdioServerParameters(
        command=sys.executable,
        args=["mcp_server_stdio.py"]
    )

    async with stdio_client(server_params) as (read_stream, write_stream):

        async with Client(read_stream, write_stream) as client:
            await client.initialize()

            result = await client.call_tool(
                "get_song_recommendation",
                {}
            )

            for content in result.content:
                print(content.text)

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
@mcp.prompt()
def get_movie_showtimes(
    city: str,
    movie_name: str,
    date: str
) -> str:
    return f"""
Find available showtimes for:

Movie: {movie_name}
City: {city}
Date: {date}

Return:
- Theatre name
- Available showtimes
- Available seats

Do not book tickets without explicit user confirmation.
"""